# 03 — ADME Experiments: Learning Curves & Noise Injection

**Goal**: Quantify how dataset size and label noise affect ML model performance for ADME endpoints.
**Phases**: 4 (learning curves), 5 (noise injection), 5b (validation quality), 6 (2D grid).
**Upstream**: Baseline models and tuning pipeline from `01_adme_eda_baseline.ipynb`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.features import morgan_fingerprints
from src.cleaning import filter_endpoint
from src.models import get_baseline_models, evaluate_model
from src.tuning import tune_lightgbm, tune_rf
from src.noise import add_gaussian_noise, add_systematic_bias, add_gross_errors

SEED = 42
DATA_PATH = '../data/raw/ADME_public_set_3521.csv'

# ── Run flags ──
RUN_LEARNING_CURVES = True
RUN_NOISE_INJECTION = True
RUN_MPNN2_EXPERIMENTS = True  # expensive — set True to include MPNN2

# ── Experiment config ──
FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75, 1.0]
N_SEEDS = 10
TUNE_N_ITER = 50       # match notebook 01
TUNE_VAL_FRAC = 0.2    # val fraction for tune_* — matches notebook 01

# ── Noise levels ──
GAUSSIAN_SIGMAS = [0.0, 0.1, 0.3, 0.5, 1.0]      # fraction of endpoint std
BIAS_FRACS = [0.0, 0.1, 0.3, 0.5, 1.0]             # fraction of endpoint std
GROSS_ERROR_FRACS = [0.0, 0.01, 0.05, 0.10, 0.20]  # fraction of labels replaced

# ── Endpoints (modelling subset — PPB excluded) ──
MODEL_ENDPOINTS = [
    'LOG HLM_CLint (mL/min/kg)',
    'LOG MDR1-MDCK ER (B-A/A-B)',
    'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'LOG RLM_CLint (mL/min/kg)',
]
EP_SHORT = ['HLM', 'MDR1', 'SOL', 'RLM']
EP_SHORT_MAP = dict(zip(MODEL_ENDPOINTS, EP_SHORT))

print('Imports OK')

## 1. Data Loading & Preparation

Load ADME dataset, filter per endpoint, featurize with ECFP4, create fixed train/test splits.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} compounds')

# Prepare per-endpoint data: fixed train/test split (same as notebook 01)
ep_data = {}
for col in MODEL_ENDPOINTS:
    df_ep = filter_endpoint(df, col)
    X = morgan_fingerprints(df_ep['SMILES'].tolist())
    y = df_ep[col].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )
    ep_data[col] = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
    }
    ep_short = EP_SHORT_MAP[col]
    print(f'  {ep_short}: {len(X_train)} train, {len(X_test)} test')

print('Data prepared.')

## 2. Learning Curves (Phase 4)

Vary training set size at zero noise. Two arms:
- **Baseline**: default hyperparameters
- **Tuned**: re-tune LightGBM and RF via RandomizedSearchCV at each fraction

BayesianRidge is baseline-only (no tuning). MeanPredictor included as reference floor.
Fractions: {0.05, 0.10, 0.25, 0.50, 0.75, 1.0}, 10 seeds each.

In [ ]:
if not RUN_LEARNING_CURVES:
    print('Learning curves skipped (RUN_LEARNING_CURVES=False).')
    lc_results = pd.DataFrame()
else:
    from sklearn.linear_model import BayesianRidge
    from sklearn.dummy import DummyRegressor
    from lightgbm import LGBMRegressor
    from sklearn.ensemble import RandomForestRegressor
    from src.tuning import make_model

    lc_rows = []
    total = len(MODEL_ENDPOINTS) * len(FRACTIONS) * N_SEEDS
    step = 0

    for col in MODEL_ENDPOINTS:
        ep_short = EP_SHORT_MAP[col]
        d = ep_data[col]
        X_train, X_test = d['X_train'], d['X_test']
        y_train, y_test = d['y_train'], d['y_test']

        for frac in FRACTIONS:
            # ── Tune ONCE per (endpoint, fraction) using a fixed reference subsample ──
            n_sub_ref = max(int(len(X_train) * frac), 2)
            idx_ref = np.random.RandomState(99).choice(len(X_train), n_sub_ref, replace=False)
            X_ref, y_ref = X_train[idx_ref], y_train[idx_ref]

            # Split ref into tune-train / tune-val — same pattern as notebook 01
            X_tune_r, X_val_r, y_tune_r, y_val_r = train_test_split(
                X_ref, y_ref, test_size=TUNE_VAL_FRAC, random_state=42
            )
            _, lgbm_params = tune_lightgbm(X_tune_r, y_tune_r, X_val_r, y_val_r,
                                           n_iter=TUNE_N_ITER, random_state=42)
            _, rf_params = tune_rf(X_tune_r, y_tune_r, X_val_r, y_val_r,
                                   n_iter=TUNE_N_ITER, random_state=42)
            print(f'  Tuned {ep_short} frac={frac}')

            for seed in range(N_SEEDS):
                step += 1
                n_sub = max(int(len(X_train) * frac), 2)
                rng = np.random.RandomState(seed)
                idx = rng.choice(len(X_train), n_sub, replace=False)
                X_sub, y_sub = X_train[idx], y_train[idx]

                base = {
                    'endpoint': col, 'ep_short': ep_short,
                    'fraction': frac, 'n_train': n_sub, 'seed': seed,
                }

                # ── Baseline arm ──
                baseline_models = {
                    'MeanPredictor': DummyRegressor(strategy='mean'),
                    'BayesianRidge': BayesianRidge(),
                    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1),
                    'LightGBM': LGBMRegressor(n_estimators=100, random_state=seed, verbose=-1),
                }
                for name, model in baseline_models.items():
                    model.fit(X_sub, y_sub)
                    metrics = evaluate_model(model, X_test, y_test)
                    lc_rows.append({**base, 'model': name, 'arm': 'baseline', **metrics})

                # ── Tuned arm: build from frozen params, fit on this seed's subsample ──
                tuned_lgbm = make_model('LightGBM', lgbm_params)
                tuned_lgbm.fit(X_sub, y_sub)
                metrics = evaluate_model(tuned_lgbm, X_test, y_test)
                lc_rows.append({**base, 'model': 'LightGBM', 'arm': 'tuned', **metrics})

                tuned_rf = make_model('RandomForest', rf_params)
                tuned_rf.fit(X_sub, y_sub)
                metrics = evaluate_model(tuned_rf, X_test, y_test)
                lc_rows.append({**base, 'model': 'RandomForest', 'arm': 'tuned', **metrics})

                if step % 20 == 0 or step == total:
                    print(f'    [{step}/{total}] {ep_short} frac={frac} seed={seed}')

    lc_results = pd.DataFrame(lc_rows)
    print(f'\nLearning curves done: {len(lc_results)} result rows')

### 2.1 Learning Curve Summary

In [ ]:
if lc_results.empty:
    print('No learning curve results.')
else:
    # Mean R2 per (endpoint, model, arm, fraction)
    summary = lc_results.groupby(['ep_short', 'model', 'arm', 'fraction'])['R2'].agg(['mean', 'std']).reset_index()
    summary.columns = ["ep_short", "model", "arm", "fraction", "R2_mean", "R2_std"]
    print(summary.to_string(index=False))

### 2.2 Learning Curve Plots

R2 vs training fraction, one subplot per endpoint. Solid = tuned, dashed = baseline.

In [ ]:
if lc_results.empty:
    print('No learning curve results to plot.')
else:
    MODEL_COLORS = {'LightGBM': '#4878cf', 'RandomForest': '#4caf6e',
                    'BayesianRidge': '#e8a83e', 'MeanPredictor': '#999999'}

    fig, axes = plt.subplots(1, len(EP_SHORT), figsize=(5 * len(EP_SHORT), 5), sharey=False)
    if len(EP_SHORT) == 1:
        axes = [axes]

    for ax, ep in zip(axes, EP_SHORT):
        ep_df = lc_results[lc_results["ep_short"] == ep]

        for model in ["LightGBM", "RandomForest", "BayesianRidge", "MeanPredictor"]:
            color = MODEL_COLORS.get(model, "grey")

            # Baseline arm
            bl = ep_df[(ep_df["model"] == model) & (ep_df["arm"] == "baseline")]
            if not bl.empty:
                agg = bl.groupby("fraction")["R2"].agg(["mean", "std"])
                ax.errorbar(agg.index, agg["mean"], yerr=agg["std"],
                            color=color, linestyle="--", marker="o", markersize=4,
                            label=f"{model} (baseline)", alpha=0.7, capsize=3)

            # Tuned arm
            tn = ep_df[(ep_df["model"] == model) & (ep_df["arm"] == "tuned")]
            if not tn.empty:
                agg = tn.groupby("fraction")["R2"].agg(["mean", "std"])
                ax.errorbar(agg.index, agg["mean"], yerr=agg["std"],
                            color=color, linestyle="-", marker="s", markersize=4,
                            label=f"{model} (tuned)", capsize=3)

        ax.set_title(ep)
        ax.set_xlabel("Training fraction")
        ax.set_ylabel("R2")
        ax.axhline(0, color="black", linewidth=0.5, linestyle=":")
        ax.legend(fontsize=7, loc="lower right")
        ax.grid(True, alpha=0.3)

    fig.suptitle('Learning Curves: R2 vs Training Fraction (baseline vs tuned)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3. Noise Injection (Phase 5)

Inject label noise into training data at full N. Test set always clean.
Three noise types:
1. **Gaussian** (intra-assay): y + N(0, sigma * std)
2. **Systematic bias** (inter-assay): shift 50% of labels by c * std
3. **Gross errors** (annotation): replace k% of labels with uniform random

Two arms: baseline (default params) vs tuned (re-tune on noisy data).

In [ ]:
if not RUN_NOISE_INJECTION:
    print('Noise injection skipped (RUN_NOISE_INJECTION=False).')
    noise_results = pd.DataFrame()
else:
    from sklearn.linear_model import BayesianRidge
    from sklearn.dummy import DummyRegressor
    from lightgbm import LGBMRegressor
    from sklearn.ensemble import RandomForestRegressor
    from src.tuning import make_model

    noise_rows = []

    # Build noise conditions: (noise_type, level, noise_fn)
    noise_conditions = []
    for sigma in GAUSSIAN_SIGMAS:
        noise_conditions.append(('gaussian', sigma,
            lambda y, s, rs: add_gaussian_noise(y, s, random_state=rs)))
    for bias in BIAS_FRACS:
        noise_conditions.append(('systematic_bias', bias,
            lambda y, s, rs: add_systematic_bias(y, s, random_state=rs)))
    for ef in GROSS_ERROR_FRACS:
        noise_conditions.append(('gross_error', ef,
            lambda y, s, rs: add_gross_errors(y, s, random_state=rs)))

    total = len(MODEL_ENDPOINTS) * len(noise_conditions) * N_SEEDS
    step = 0

    for col in MODEL_ENDPOINTS:
        ep_short = EP_SHORT_MAP[col]
        d = ep_data[col]
        X_train, X_test = d['X_train'], d['X_test']
        y_train, y_test = d['y_train'], d['y_test']

        for noise_type, noise_level, noise_fn in noise_conditions:
            # ── Tune ONCE per (endpoint, noise_condition) using reference noisy labels ──
            y_ref_noisy = noise_fn(y_train, noise_level, 99)
            X_tune_r, X_val_r, y_tune_r, y_val_r = train_test_split(
                X_train, y_ref_noisy, test_size=TUNE_VAL_FRAC, random_state=42
            )
            _, lgbm_params = tune_lightgbm(X_tune_r, y_tune_r, X_val_r, y_val_r,
                                           n_iter=TUNE_N_ITER, random_state=42)
            _, rf_params = tune_rf(X_tune_r, y_tune_r, X_val_r, y_val_r,
                                   n_iter=TUNE_N_ITER, random_state=42)
            print(f'  Tuned {ep_short} {noise_type} level={noise_level}')

            for seed in range(N_SEEDS):
                step += 1
                y_noisy = noise_fn(y_train, noise_level, seed)

                base = {
                    'endpoint': col, 'ep_short': ep_short,
                    'noise_type': noise_type, 'noise_level': noise_level, 'seed': seed,
                }

                # ── Baseline arm ──
                baseline_models = {
                    'MeanPredictor': DummyRegressor(strategy='mean'),
                    'BayesianRidge': BayesianRidge(),
                    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1),
                    'LightGBM': LGBMRegressor(n_estimators=100, random_state=seed, verbose=-1),
                }
                for name, model in baseline_models.items():
                    model.fit(X_train, y_noisy)
                    metrics = evaluate_model(model, X_test, y_test)
                    noise_rows.append({**base, 'model': name, 'arm': 'baseline', **metrics})

                # ── Tuned arm: build from frozen params, fit on this seed's noisy labels ──
                tuned_lgbm = make_model('LightGBM', lgbm_params)
                tuned_lgbm.fit(X_train, y_noisy)
                metrics = evaluate_model(tuned_lgbm, X_test, y_test)
                noise_rows.append({**base, 'model': 'LightGBM', 'arm': 'tuned', **metrics})

                tuned_rf = make_model('RandomForest', rf_params)
                tuned_rf.fit(X_train, y_noisy)
                metrics = evaluate_model(tuned_rf, X_test, y_test)
                noise_rows.append({**base, 'model': 'RandomForest', 'arm': 'tuned', **metrics})

                if step % 20 == 0 or step == total:
                    print(f'    [{step}/{total}] {ep_short} {noise_type} level={noise_level} seed={seed}')

    noise_results = pd.DataFrame(noise_rows)
    print(f'\nNoise injection done: {len(noise_results)} result rows')

### 3.1 Noise Injection Summary

In [ ]:
if noise_results.empty:
    print('No noise results.')
else:
    for nt in ['gaussian', 'systematic_bias', 'gross_error']:
        print(f'
=== {nt.upper()} ===')
        sub = noise_results[noise_results['noise_type'] == nt]
        summary = sub.groupby(['ep_short', 'model', 'arm', 'noise_level'])['R2'].agg(['mean', 'std']).reset_index()
        summary.columns = ["ep_short", "model", "arm", "noise_level", "R2_mean", "R2_std"]
        print(summary.to_string(index=False))

### 3.2 Noise Degradation Plots

R2 vs noise level, one row per noise type, one column per endpoint.

In [ ]:
if noise_results.empty:
    print('No noise results to plot.')
else:
    MODEL_COLORS = {'LightGBM': '#4878cf', 'RandomForest': '#4caf6e',
                    'BayesianRidge': '#e8a83e', 'MeanPredictor': '#999999'}
    noise_types = ['gaussian', 'systematic_bias', 'gross_error']

    fig, axes = plt.subplots(len(noise_types), len(EP_SHORT),
                             figsize=(5 * len(EP_SHORT), 4 * len(noise_types)), sharey=False)

    for row, nt in enumerate(noise_types):
        nt_df = noise_results[noise_results['noise_type'] == nt]
        for col_idx, ep in enumerate(EP_SHORT):
            ax = axes[row, col_idx]
            ep_df = nt_df[nt_df["ep_short"] == ep]

            for model in ["LightGBM", "RandomForest", "BayesianRidge", "MeanPredictor"]:
                color = MODEL_COLORS.get(model, "grey")

                bl = ep_df[(ep_df["model"] == model) & (ep_df["arm"] == "baseline")]
                if not bl.empty:
                    agg = bl.groupby("noise_level")["R2"].agg(["mean", "std"])
                    ax.errorbar(agg.index, agg["mean"], yerr=agg["std"],
                                color=color, linestyle="--", marker="o", markersize=4,
                                label=f"{model} (bl)", alpha=0.7, capsize=3)

                tn = ep_df[(ep_df["model"] == model) & (ep_df["arm"] == "tuned")]
                if not tn.empty:
                    agg = tn.groupby("noise_level")["R2"].agg(["mean", "std"])
                    ax.errorbar(agg.index, agg["mean"], yerr=agg["std"],
                                color=color, linestyle="-", marker="s", markersize=4,
                                label=f"{model} (tuned)", capsize=3)

            ax.set_title(f"{ep} — {nt}")
            ax.set_xlabel("Noise level")
            ax.set_ylabel("R2")
            ax.axhline(0, color="black", linewidth=0.5, linestyle=":")
            ax.grid(True, alpha=0.3)
            if row == 0 and col_idx == 0:
                ax.legend(fontsize=6, loc="lower left")

    fig.suptitle('Noise Degradation: R2 vs Noise Level (baseline vs tuned)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 4. Save Results

In [ ]:
# Save raw results for Phase 6 surface plots
import os
os.makedirs('../data/processed', exist_ok=True)

if not lc_results.empty:
    lc_results.to_csv('../data/processed/learning_curve_results.csv', index=False)
    print(f'Saved learning curve results: {len(lc_results)} rows')

if not noise_results.empty:
    noise_results.to_csv('../data/processed/noise_injection_results.csv', index=False)
    print(f'Saved noise injection results: {len(noise_results)} rows')